**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../../docs/METHODS.md) for the implemented protocol.


# Level 1 V2 — Clip-level label splits

Same task as V1 (4-way taxonomic SSL with label propagation), but the
labeled / unlabeled split is now done at the **clip level**: every window
from a given clip is either entirely labeled or entirely unlabeled. This
closes the V1 leakage where 5 nearly-identical windows from one recording
could be split 3 labeled / 2 unlabeled, letting the graph propagate the
label trivially through the sister windows.

Pipeline:
1. Load `Level1/Data/level1_features.npz` (the V1 .npz; features unchanged).
2. k-NN graph k=10, σ=median.
3. **Clip-level** stratified 50% split, 20 trials.
4. Compare V1 (window-split) vs V2 (clip-split) accuracy.
5. Save results to `v2/Level1/level1_results_v2.json`.


In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
# Keep PROJECT_ROOT first on sys.path so `import utils.x` finds the original
# V1 utils (not v2/utils, which would cause a circular import in v2 modules).
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import build_knn_graph, label_propagation
from utils.evaluation import plot_confusion_matrix
from v2.utils.splits import stratified_clip_mask
from v2.utils.runs import make_run_dir

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level1"
DATA_DIR = LEVEL_DIR / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Per-run output folders. Each notebook execution creates a fresh
# runs/runN with its own figures/ and results/. Edit the notebook
# between runs to compare configurations.
RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level1_classification_v2.ipynb")
FIG_DIR = RUN_DIR / "figures"
RES_DIR = RUN_DIR / "results"
print(f"this run -> {RUN_DIR}")

V1_DATA = PROJECT_ROOT / "Level1" / "Data" / "level1_features.npz"


## Load features (unchanged from V1)


In [ ]:
data = np.load(V1_DATA, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"].astype(np.int64)
label_names = list(data["label_names"])
is_train = data["is_train"]

print(f"X: {X.shape}, labels: {y.shape}, clips: {len(np.unique(clip_ids))}")
print("classes:")
for u, c in zip(*np.unique(y, return_counts=True)):
    print(f"  {label_names[u]}: {c} windows")

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)


## Build k-NN graph (default V1 settings)


In [ ]:
W = build_knn_graph(Xz, k=10, sigma="median")
print(f"W: {W.shape}, nnz={W.nnz}")


## Window-level split (V1 baseline) vs Clip-level split (V2)

We run both and compare to make the leakage effect concrete. The window-
level numbers reproduce V1's setup directly so the V1 vs V2 delta is
measured in the same code path.


In [ ]:
N_TRIALS = 20
LABEL_FRAC = 0.5

def stratified_window_mask(y, frac, rng):
    mask = np.zeros(y.shape[0], dtype=bool)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        n = max(1, int(round(frac * idx.size)))
        chosen = rng.choice(idx, size=n, replace=False)
        mask[chosen] = True
    return mask

def trial(W, mask):
    pred = label_propagation(W, y, mask)
    unl = ~mask
    return (
        accuracy_score(y[unl], pred[unl]),
        f1_score(y[unl], pred[unl], average="macro",
                 labels=range(len(label_names)), zero_division=0),
        pred,
    )

window_acc, window_f1 = [], []
clip_acc, clip_f1 = [], []
clip_preds = []
clip_masks = []

t0 = time.time()
for t in range(N_TRIALS):
    rng_w = np.random.default_rng(1000 + t)
    rng_c = np.random.default_rng(2000 + t)

    m_w = stratified_window_mask(y, LABEL_FRAC, rng_w)
    a_w, f_w, _ = trial(W, m_w)
    window_acc.append(a_w); window_f1.append(f_w)

    m_c = stratified_clip_mask(clip_ids, y, LABEL_FRAC, rng_c)
    a_c, f_c, p_c = trial(W, m_c)
    clip_acc.append(a_c); clip_f1.append(f_c)
    clip_preds.append(p_c); clip_masks.append(m_c)

print(f"\n20 trials done in {time.time() - t0:.1f}s\n")
print(f"Window-split (V1):  acc {np.mean(window_acc):.4f} ± {np.std(window_acc):.4f}, "
      f"macro F1 {np.mean(window_f1):.4f} ± {np.std(window_f1):.4f}")
print(f"Clip-split   (V2):  acc {np.mean(clip_acc):.4f} ± {np.std(clip_acc):.4f}, "
      f"macro F1 {np.mean(clip_f1):.4f} ± {np.std(clip_f1):.4f}")


## Confusion matrix on a representative V2 clip-level trial


In [ ]:
unl0 = ~clip_masks[0]
plot_confusion_matrix(
    y[unl0], clip_preds[0][unl0], label_names,
    save_path=FIG_DIR / "confusion_matrix_label_prop_v2.png",
    title="Level 1 V2 — label prop, clip-level split (trial 1)",
)


## Persist results so Fix 2 / final summary can reuse them


In [ ]:
def report_clip_split_stats(clip_ids, y):
    uniq, counts = np.unique(clip_ids, return_counts=True)
    return {
        "n_clips": int(uniq.size),
        "n_windows": int(y.size),
        "windows_per_clip_mean": float(counts.mean()),
        "windows_per_clip_max": int(counts.max()),
    }

results = {
    "label_fraction": LABEL_FRAC,
    "n_trials": N_TRIALS,
    "window_split_v1": {
        "accuracy_mean": float(np.mean(window_acc)),
        "accuracy_std": float(np.std(window_acc)),
        "f1_mean": float(np.mean(window_f1)),
        "f1_std": float(np.std(window_f1)),
    },
    "clip_split_v2": {
        "accuracy_mean": float(np.mean(clip_acc)),
        "accuracy_std": float(np.std(clip_acc)),
        "f1_mean": float(np.mean(clip_f1)),
        "f1_std": float(np.std(clip_f1)),
    },
    "data": report_clip_split_stats(clip_ids, y),
}
out_json = RES_DIR / "level1_results_v2.json"
out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("wrote", out_json)

# Also persist the trial-0 clip mask so the CNN baseline (Fix 2) can use the
# exact same labeled subset. Saved both into this run's results folder
# (snapshot) and into Level1/Data/ (where the latest run's mask lives, so
# the resnet notebook can find it without knowing the run number).
mask_payload = dict(
    mask=clip_masks[0],
    is_train=is_train,
    clip_ids=clip_ids,
)
np.savez_compressed(RES_DIR / "level1_v2_trial0_mask.npz", **mask_payload)
np.savez_compressed(DATA_DIR / "level1_v2_trial0_mask.npz", **mask_payload)
print("wrote mask to", RES_DIR / "level1_v2_trial0_mask.npz")
print("also cached at", DATA_DIR / "level1_v2_trial0_mask.npz")
